# test bigquery

In [1]:
import os, sys
import pandas as pd
from dotenv import load_dotenv
from google.cloud import bigquery

/home/charourou/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.6) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
sys.path

['/home/charourou/projects/velonantes',
 '/home/charourou/code/charourou/03-Decision-Science',
 '/home/charourou/projects/Projet_Hydrosense/notebooks/old',
 '/home/charourou/.pyenv/versions/3.10.6/lib/python310.zip',
 '/home/charourou/.pyenv/versions/3.10.6/lib/python3.10',
 '/home/charourou/.pyenv/versions/3.10.6/lib/python3.10/lib-dynload',
 '',
 '/home/charourou/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages']

In [2]:
import google.auth
credentials, project = google.auth.default()
print("Compte utilisé par Python :", credentials.service_account_email if hasattr(credentials, 'service_account_email') else "Compte utilisateur (gcloud)")
print("Projet détecté par défaut :", project)

Compte utilisé par Python : hydrosense-bigquery@hydro-sense-498112.iam.gserviceaccount.com
Projet détecté par défaut : hydro-sense-498112


In [3]:
# 1 Connexion

# 1.1 Charge les variables du fichier .env
load_dotenv(dotenv_path="../.env")

# 1.2. Récupère la valeur de la variable d'environnement
GCP_PROJECT_ID = os.getenv("GCP_PROJECT_ID")

# 1.3. Initialise le client avec la variable définie
client = bigquery.Client(project=GCP_PROJECT_ID)

print(f"Connecté à {GCP_PROJECT_ID} ✅")

Connecté à hydro-sense-498112 ✅


In [4]:
# 2 Load test model "piezo_test"

print(os.getenv("GOOGLE_APPLICATION_CREDENTIALS"))
print(os.getenv("GCP_PROJECT_ID"))

DATASET = os.getenv("BQ_DATASET_ID")
TABLE = "piezo_test"

query = f"""
    SELECT *
    FROM {GCP_PROJECT_ID}.{DATASET}.{TABLE}
    LIMIT 100
"""

query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()
df.head()

/home/yanns/code/Yannsimon92/100-Hydrosense/Projet_Hydrosense/secrets/hydro-sense-498112-d8b48c4804b5.json
hydro-sense-498112


/home/yanns/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,string_field_0
0,date_mesure;niveau_nappe_eau;profondeur_nappe
1,1990-09-03;31.85;8.25
2,1990-11-13;30.95;9.15
3,1991-02-21;33.54;6.56
4,1991-05-13;33.55;6.55


In [ ]:
# problem of parsing ?? régler plus tard ??

In [ ]:
# 3 test uploadinf

df_test = pd.read_csv(f'../../raw_data/piezo_BSS001BLTP.csv', sep = ';')
df_test.head()

,date_mesure,niveau_nappe_eau,profondeur_nappe
0,1994-04-22,44.19,44.19
1,1994-04-23,44.29,44.29
2,1994-04-24,44.26,44.26
3,1994-04-25,44.22,44.22
4,1994-04-26,44.19,44.19


In [6]:
TABLE = "piezo_upload_test"

table = f"{GCP_PROJECT_ID}.{DATASET}.{TABLE}"

client = bigquery.Client()

write_mode = "WRITE_TRUNCATE" # or "WRITE_APPEND"
job_config = bigquery.LoadJobConfig(write_disposition=write_mode)

job = client.load_table_from_dataframe(df_test, table, job_config=job_config)
result = job.result()

/home/yanns/.pyenv/versions/Projet_Hydrosense/lib/python3.10/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Forbidden: 403 POST https://bigquery.googleapis.com/upload/bigquery/v2/projects/wagon-bootcamp-493209-d3/jobs?uploadType=multipart: Access Denied: Dataset hydro-sense-498112:piezometry: Permission bigquery.tables.create denied on dataset hydro-sense-498112:piezometry (or it may not exist).

In [14]:
TABLE = "piezo_upload_test"

query = f"""
    SELECT *
    FROM {GCP_PROJECT_ID}.{DATASET}.{TABLE}
    LIMIT 100
"""

query_job = client.query(query)
result = query_job.result()
df = result.to_dataframe()
df.head()

/home/yanns/.pyenv/versions/3.10.6/envs/Projet_Hydrosense/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date_mesure,niveau_nappe_eau,profondeur_nappe
0,2005-10-13,36.88,36.88
1,2005-10-12,36.90,36.90
2,2005-10-09,36.92,36.92
3,2005-10-11,36.92,36.92
4,2005-10-10,36.94,36.94
